# MyoChallenge 2026 — ChaseTag quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/gist/Vittorio-Caggiano/a47d35329bb4e89836546b5e50017d65/ChaseTag_Colab_Quickstart.ipynb)

For **first-time invitees**. About 10 minutes:

1. Register on the **website** (not in this notebook)
2. Copy your **CLI token** from Submissions
3. Run the cells here to upload a dummy model
4. Watch the result on the **website**

Site: [mc26.myosuite.org](https://mc26.myosuite.org)

Invitation only. If the site says your email is not on the list, [apply here](https://docs.google.com/forms/d/e/1FAIpQLSf0tsCktz3adRUB6YGd0OfQMP9lQDrzeHeZCozrlVDN2J6_qw/viewform) first.


## What you submit

Each team submits **one file: a single player**.

You do **not** submit an opponent. The server always provides the other player.

| You send | The server does |
|---|---|
| **One** ONNX model | Runs it as player A |
| | Puts a champion (or another team) as player B |
| | Scores the match |

The file is a neural net that, every step:

- **reads** 546 numbers — your body, the opponent’s **hip** (not their muscles), and a **chase/evade flag**
- **writes** 354 numbers — your muscle activations, each between 0 and 1

```
obs (546)  →  YOUR MODEL  →  action (354)
```

You do **not** pick chase vs evade. One file plays both. The server assigns the role each episode and writes it into the observation. Opponent hip is only **where they are** — not whether to hunt or flee.

| Slice | What it is |
|---|---|
| `obs[0:537]` | Your body (joints, velocities, muscles, pelvis) |
| `obs[537:544]` | Opponent hip relative pose (where they are) |
| `obs[544:546]` | **Role** — `[1, 0]` = **chase** (go toward them), `[0, 1]` = **evade** (run away) |

Do not infer chase/evade from hip position. The same offset means opposite actions depending on those last two numbers.

That same file is ranked against a fixed champion during the cycle, then used in a 1v1 tournament against other teams.


## 1. Register on the website

Do this in the browser, not in Colab.

1. Open [mc26.myosuite.org](https://mc26.myosuite.org).
2. Click **Register your team** (or the **Register** tab).
3. Fill **team name**, **email**, and **password** (8+ characters). Country and education are optional.
4. Click **Register team**.

Already registered? Use **Sign in** instead.

![Register tab](https://gist.githubusercontent.com/Vittorio-Caggiano/a47d35329bb4e89836546b5e50017d65/raw/colab-register.png)

![Team name and email](https://gist.githubusercontent.com/Vittorio-Caggiano/a47d35329bb4e89836546b5e50017d65/raw/colab-register-form.png)

![Register team button](https://gist.githubusercontent.com/Vittorio-Caggiano/a47d35329bb4e89836546b5e50017d65/raw/colab-register-submit.png)

![Sign in](https://gist.githubusercontent.com/Vittorio-Caggiano/a47d35329bb4e89836546b5e50017d65/raw/colab-signin.png)


## 2. Copy your CLI token

Still on the website:

1. Click **Submissions** in the top bar.
2. Copy the **CLI submission token** (green label).

You will paste **email + this token** into the Colab cell below. Do not share the token.

![CLI token and submissions](https://gist.githubusercontent.com/Vittorio-Caggiano/a47d35329bb4e89836546b5e50017d65/raw/colab-submissions.png)


## 3. Upload a dummy model from this notebook

This dummy always outputs `0.5`. It only checks that submit works. It is **not** a ChaseTag player.

Paste the email and token from the website, then **Runtime → Run all** (or run the next three cells).


In [ ]:
# @title Install
%pip -q install -U "evalmc>=0.0.4" onnx numpy
print("ok")


In [ ]:
# @title Paste email + CLI token from the website
EMAIL = ""  # @param {type:"string"}
AUTH_TOKEN = ""  # @param {type:"string"}

EMAIL = EMAIL.strip().lower()
AUTH_TOKEN = AUTH_TOKEN.strip()
if not EMAIL or not AUTH_TOKEN:
    raise SystemExit("Paste EMAIL and AUTH_TOKEN from mc26.myosuite.org → Submissions.")
print("email", EMAIL, "token length", len(AUTH_TOKEN))


In [ ]:
# @title Build a dummy ONNX (not a real player)
from pathlib import Path

import numpy as np
import onnx
from onnx import TensorProto, helper, numpy_helper

OBS, ACT = 546, 354
ONNX_PATH = Path("chasetag_smoke.onnx")

w = numpy_helper.from_array(np.zeros((OBS, ACT), dtype=np.float32), name="W")
b = numpy_helper.from_array(np.full(ACT, 0.5, dtype=np.float32), name="b")
obs = helper.make_tensor_value_info("obs", TensorProto.FLOAT, [None, OBS])
action = helper.make_tensor_value_info("action", TensorProto.FLOAT, [None, ACT])
graph = helper.make_graph(
    [helper.make_node("Gemm", ["obs", "W", "b"], ["action"])],
    "chasetag_dummy",
    [obs],
    [action],
    [w, b],
)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 13)], producer_name="mc26-colab")
model.ir_version = 8  # Compatible with the server ONNX Runtime
onnx.checker.check_model(model)
onnx.save(model, str(ONNX_PATH))
print(f"Wrote {ONNX_PATH.resolve()}")
print(f"Shape: obs [batch, {OBS}] → action [batch, {ACT}]")


In [ ]:
# @title Submit
import subprocess
import sys

cmd = [
    sys.executable, "-m", "evalmc", "submit-onnx", str(ONNX_PATH),
    "-t", "chasetag", "-p", "1",
    "-e", EMAIL, "-a", AUTH_TOKEN,
]
print("Uploading", ONNX_PATH.name, "…")
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise SystemExit(proc.returncode)
print("Upload accepted. Now open the website Submissions tab (next section).")


## 4. Watch evaluation on the website

Go back to [mc26.myosuite.org](https://mc26.myosuite.org) → **Submissions**. Refresh until the row moves:

| Status | Meaning |
|---|---|
| **submitted** | queued |
| **evaluating** | running |
| **scored** | done — a number appears under Score |
| **failed** | click the row for the error |

![Watch status on Submissions](https://gist.githubusercontent.com/Vittorio-Caggiano/a47d35329bb4e89836546b5e50017d65/raw/colab-submissions.png)

The dummy `0.5` model is only a pipeline check. Train a real `546 → 354` player before freeze, then run the **Submit** cell on that file instead.


## After this works: train a real player

Still **one file**. Train a net `obs (546) → action (354)`, export ONNX, submit the same way.

1. Train a walker in [MyoSuite tutorial 5.4](https://colab.research.google.com/github/MyoHub/myosuite/blob/main/tutorials/5.4_MuscleMimic_Directional_Locomotion.ipynb).
2. Use the opponent hip position in `obs` to choose a heading (toward them to chase, away to flee).
3. Practice against the scripted opponent in `myoChallengeChaseTagFBP2-v0`.
4. Export ONNX with input `obs` `[batch, 546]` and output `action` `[batch, 354]`.

Do not export the 5.4 walker as-is: it expects 528 numbers; the server sends 546.
